# Procesamiento de datos mediante Apache Spark

## Autores

Eduardo Alarcón Navarro - 100472175@alumnos.uc3m.es

Ángela Ochoa Lavieja - 100562058@alumnos.uc3m.es

Lucía Velasco González - 100383335@alumnos.uc3m.es

Manuel Gómez-Plana Rodríguez - 100472310@alumnos.uc3m.es

# 1. Introducción

Este documento guarda la información relacionada con la elaboración de la práctica 1 de la asignatura Computación de Altas Prestaciones del Máster Universitario en Ciencia y Tecnología Informática. El notebook se divide en las siguientes secciones:

1. **Introducción**: se introduce la práctica.
2. **Problemas**: se introducen los posibles problemas planteados para realizar el estudio y el problema escogido para su análisis.
3. **Resolución**: se resuelve el problema mediante tres técnicas distintas: consultas SQL, concatenación de dataframes de Pandas y uso de RDDs.
4. **Análisis**: se analizan los resultados obtenidos de los tres problemas para determinar aspectos como la solución más rápida, el speedup obtenido mediante cada técnica o la escalabilidad de las soluciones.
5. **Conclusiones**: se concluye el trabajo con unos pensamientos finales.

# Problemas



## Problemas Planteados

## Problema escogido

# Resolución

In [ ]:
# Importado de Spark
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql.functions import *

# Creación de la sesión de spark
spark = SparkSession.builder.master('local[*]').getOrCreate()
#spark.conf.set('spark.cores.max', '1')
#spark.conf.set('spark.driver.memory','1000M')

sc = spark.sparkContext
sc

In [ ]:
zone_map = spark.read.format("csv").option("inferSchema", "true").option("header", "true").option("mode", "DROPMALFORMED").load("content/taxi_zone_lookup.csv")
zone_map.show()
zone_map: dict[int, list[str]] = zone_map.toPandas().set_index('LocationID').T.to_dict('list')

In [ ]:
df = spark.read.format("parquet").option("inferSchema", "true").option("timestampFormat","yyyy-MM-dd HH:mm:ss").option("header", "true").option("mode", "DROPMALFORMED").load("content/yellow_tripdata_2025-01.parquet")
df.printSchema()
df.show(5)
type(df)

Limpieza de datos

In [ ]:
# df = df.na.drop()

In [ ]:
df = df.filter(df.trip_distance != 0)
df = df.filter(df.tpep_pickup_datetime != df.tpep_dropoff_datetime)

In [ ]:
TABLE_NAME = 'taxi_routes'
df.createOrReplaceTempView(TABLE_NAME)
SQL = f"""SELECT HOUR(tpep_pickup_datetime) AS hour,
            ROUND(AVG(trip_distance / DATE_DIFF(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime))*1.6*3600, 2) as vel_media_km_h 
            FROM {TABLE_NAME} 
            GROUP BY hour 
            SORT BY hour """
sql_result = spark.sql(SQL)

In [ ]:
sql_result.show(24)

## Usando dataframes

In [ ]:
df2=df.withColumn('DiffInSeconds', (col('tpep_dropoff_datetime') - col('tpep_pickup_datetime')).cast('long'))
df3 = df2.withColumn('vel_km_h', try_divide('trip_distance','DiffInSeconds')*5760)
df4 = df3.withColumn('start_hour', hour(col('tpep_pickup_datetime')))
df4.select('start_hour', 'vel_km_h').groupBy("start_hour").avg('vel_km_h').sort('start_hour').show(truncate=False)

Usando RDD

# Análisis

# Conclusiones